In [1]:
import os
import boto3
import pickle

from tqdm import tqdm
from parameters import ES_S3_ACCESS_POINT
from earthscope_sdk import EarthScopeClient

In [2]:
client = EarthScopeClient()
cred = client.user.get_aws_credentials()
print(f"Credential expires at {cred.expiration} UTC")

Credential expires at 2025-10-14 06:45:34+00:00 UTC


In [3]:
session = boto3.Session(
    aws_access_key_id=cred.aws_access_key_id,
    aws_secret_access_key=cred.aws_secret_access_key,
    aws_session_token=cred.aws_session_token,
)

s3_client = session.client("s3")

In [ ]:
networks = [i['Prefix'].split("/")[-2] for i in s3_client.list_objects_v2(Bucket=ES_S3_ACCESS_POINT, 
                        Prefix=f"miniseed/", Delimiter="/")['CommonPrefixes']]
print(f"There are {len(networks)} networks in total")

There are 483 networks in total


In [9]:
for inet, net in enumerate(networks):
    all_lists = {}
    if os.path.exists(f"./data/{net}.pt"):
        continue
    if net in all_lists:
        pass
    else:
        all_lists[net] = {}
    years = [i['Prefix'].split("/")[-2] for i in s3_client.list_objects_v2(Bucket=ES_S3_ACCESS_POINT, 
                        Prefix=f"miniseed/{net}/", Delimiter="/")['CommonPrefixes']]
    print(f"Working on {inet}.{net}: {years[0]} to {years[-1]}")
    for yyyy in tqdm(years):
        if yyyy in all_lists[net]:
            pass
        else:
            all_lists[net][yyyy] = {}
        days = [i['Prefix'].split("/")[-2] for i in s3_client.list_objects_v2(Bucket=ES_S3_ACCESS_POINT, 
                        Prefix=f"miniseed/{net}/{yyyy}/", Delimiter="/")['CommonPrefixes']]
        for ddd in days:
            if ddd in all_lists[net][yyyy]:
                continue
            contents = s3_client.list_objects_v2(Bucket=ES_S3_ACCESS_POINT, 
                        Prefix=f"miniseed/{net}/{yyyy}/{ddd}/", Delimiter="/")['Contents']
            all_lists[net][yyyy][ddd] = [i for i in zip([i['Key'].split("/")[-1].split(".")[0] for i in contents], 
                                        [i['Size'] for i in contents])]
    with open(f"./data/{net}.pt", 'wb') as f:
        pickle.dump(all_lists, f)

Working on 0.12: 2022 to 2023


100%|██████████| 2/2 [00:08<00:00,  4.35s/it]


Working on 1.16: 2015 to 2025


100%|██████████| 10/10 [01:03<00:00,  6.35s/it]


Working on 2.17: 2023 to 2023


...

100%|██████████| 1/1 [00:02<00:00,  2.89s/it]


Working on 329.RU: 2017 to 2022


100%|██████████| 6/6 [00:43<00:00,  7.26s/it]


Working on 364.UT: 2025 to 2025


100%|██████████| 1/1 [00:05<00:00,  5.09s/it]
